# Step 09 — Single Agent

Move from a hand-written prompt to a CrewAI `Agent`. This notebook is **standalone** — the agent is defined right here in code, no separate YAML config or project files to edit. The `role`, `goal`, and `backstory` are still just a system prompt under the hood — CrewAI assembles it for you. What the framework adds is the loop: the agent reasons in steps before producing output, can call tools (Steps 11/12), and retries on failure.

## Learning objective

By the end of this notebook, you will:

- Understand what a CrewAI `Agent` is, and how it differs from the system-message-shaped prompts of Steps 02–07
- Understand the ReAct (Reason + Act) loop that lets an agent take multiple internal reasoning steps before producing a final answer
- Have defined an `Agent` and a `Task`, and run them together via a minimal `Crew` — the three building blocks every CrewAI project uses

In [1]:
import os

from dotenv import load_dotenv # loads the LLM API key from .env file
from crewai import Agent, Task, Crew, Process # import the CrewAI framework
from IPython.display import Markdown, display # display Markdown output in Jupyter Notebook

load_dotenv()

topic = "EU AI Act"

# Agent - Role, Goal, Backstory Framework
agent = Agent(
    role=f"Senior Research Analyst specializing in {topic} compliance and regulatory developments",
    goal=(
        f"Deliver accurate, well-structured explanations of {topic}'s requirements "
        "that a compliance team could act on directly, grounded in the Act's actual "
        "text rather than general AI-regulation knowledge"
    ),
    backstory=(
        f"With over a decade spent tracking EU technology regulation, you've built a "
        f"reputation for cutting through legal ambiguity and explaining what {topic} "
        "actually requires, not what commentators assume it requires. You cross-check "
        "every claim against the regulation's own text before presenting it as fact."
    ),
    verbose=False,
)

# Task - Expected Output Framework
task = Task(
    description=(
        f"Explain {topic}'s risk-based categories. Specifically:\n"
        "1. Define each category: unacceptable, high-risk, limited, minimal.\n"
        "2. For high-risk systems, list the specific obligations that apply to providers.\n"
        "3. Note which obligations are already in force and which are still upcoming."
    ),
    expected_output=(
        f"A structured explanation of {topic}'s risk categories, with one clearly "
        "labeled section per category and a dedicated section listing high-risk "
        "providers' obligations alongside their applicable dates."
    ),
    agent=agent,
)

# ── Crew — one agent, one task, the minimal case ──────────────────────────────
crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff()
display(Markdown(result.raw))

As a Senior Research Analyst, I have structured this breakdown based on the final text of Regulation (EU) 2024/1689 (the EU AI Act). The risk-based approach is the core architecture of the regulation, determining the stringency of compliance obligations based on the potential harm an AI system poses to fundamental rights and safety.

---

### 1. The Risk-Based Categories

The EU AI Act classifies AI systems into four distinct tiers based on the risk they pose:

*   **Unacceptable Risk (Prohibited):** AI systems considered a clear threat to fundamental rights, safety, or democratic values are banned. This includes systems that use subliminal techniques to manipulate behavior, exploit vulnerabilities of specific groups, social scoring by public authorities, and real-time remote biometric identification in publicly accessible spaces for law enforcement (with narrow exceptions).
*   **High-Risk:** AI systems that pose significant risks to health, safety, or fundamental rights. This category is bifurcated into (a) AI components of products already subject to EU safety legislation (e.g., toys, aviation, medical devices) and (b) specific stand-alone systems explicitly listed in Annex III (e.g., critical infrastructure, education, employment, law enforcement, and migration management).
*   **Limited Risk (Transparency):** AI systems that interact with humans (e.g., chatbots) or generate/manipulate content (e.g., deepfakes). These are subject to lighter obligations focusing on transparency—users must be informed they are interacting with an AI or that content has been synthetically altered.
*   **Minimal Risk:** The vast majority of AI systems currently in use (e.g., spam filters, inventory management, AI-enabled video games). The Act does not impose mandatory requirements on these systems, though it encourages the adoption of voluntary codes of conduct.

---

### 2. Obligations for High-Risk AI Providers

Providers of high-risk AI systems must implement a comprehensive management system to ensure compliance. The core obligations, as defined in Articles 8–27, include:

*   **Risk Management System:** A continuous, iterative process to identify, estimate, and mitigate risks throughout the system's lifecycle.
*   **Data Governance:** Requirements ensuring training, validation, and testing datasets meet specific quality criteria (e.g., relevance, representativeness, error-free, and complete).
*   **Technical Documentation:** A detailed file providing evidence that the system complies with all regulatory requirements, kept ready for competent authorities.
*   **Record-Keeping:** Automatic generation of logs (event logging) to ensure traceability of the system's functioning throughout its lifecycle.
*   **Transparency and Information for Deployers:** Providing instructions for use that are clear, concise, and allow deployers to understand the system’s capabilities and limitations.
*   **Human Oversight:** Designing the system to allow for effective supervision by human beings to prevent or minimize risks during operation.
*   **Accuracy, Robustness, and Cybersecurity:** Ensuring the system meets appropriate levels of performance, reliability, and protection against unauthorized attempts to alter its use or performance.
*   **Conformity Assessment:** Undergoing a formal assessment procedure to demonstrate compliance before the system is placed on the market or put into service.
*   **Quality Management System (QMS):** Establishing a systematic, documented approach to organizational management to ensure compliance (e.g., record-keeping, resource management).

---

### 3. Timeline and Enforcement

The EU AI Act entered into force on **August 1, 2024**. Compliance obligations are being "phased in" over the coming years:

| Milestone Date | Requirement / Obligation |
| :--- | :--- |
| **February 2, 2025** | **Prohibitions:** Systems deemed "Unacceptable Risk" must be decommissioned or removed from the EU market. |
| **August 2, 2026** | **General AI Requirements:** Obligations for General Purpose AI (GPAI) models become applicable. |
| **August 2, 2026** | **High-Risk (Annex III):** The majority of obligations for high-risk AI systems (Article 6(2) and Annex III) become applicable. |
| **August 2, 2027** | **Safety Components:** Obligations for high-risk systems that are components of products already regulated by EU safety legislation (Annex I) become applicable. |

**Important Note for Compliance Teams:** While the full compliance framework for high-risk systems activates in August 2026, the governance structure (National Competent Authorities and the AI Office) is currently being established. Organizations should utilize the next 18 months to perform "gap analyses" of their high-risk assets, as retrospective compliance with data governance and documentation requirements is significantly more resource-intensive than "compliance by design."

## Your task

1. Run the cell. Read the verbose log above the final answer — this is the first time you can see the agent's internal reasoning, not just the final answer. Does the agent break the task into sub-steps?

2. Compare the Researcher's answer to the prompting steps' (Steps 02–07) plain-prompt answer on a similar question. What does having an explicit `role`/`goal`/`backstory` add, if anything, over a system message you wrote by hand — given that CrewAI compiles them into almost the same thing under the hood (see the Background note above)?

3. Try adding a parameter from [Step 08](step_08_intro_to_crewai.ipynb)'s full reference list and rerunning: set `verbose=True` to see the reasoning log, or `max_iter=2` and see whether the agent gets cut off before finishing. Remove it afterward.

4. Swap in your own team's topic: edit the Agent's `role`/`goal`/`backstory` and the Task's `description`/`expected_output` directly in the `Agent(...)`/`Task(...)` calls. Keep that identity the same across Steps 10–14 so the comparisons stay meaningful.

5. Note what you observed — as a team, you'll draw on this for `REPORT.md`'s Section 3 (System Architecture) and 4.1 (LLM Selection & Configuration), once you start designing your own agent. This closes out Sprint 2 — add its row to the **Sprint Progression** table before your team's Interim Presentation.

## Shortcomings

One agent, however capable, can only reason over what it already knows — it has no way to check anything against the current state of the world, it forgets everything the moment one `kickoff()` call ends, and it can't be simultaneously a credulous researcher gathering everything and a skeptical analyst questioning what it found.

[Step 10](step_10_memory.ipynb) addresses the second gap first: giving the agent recall across separate calls, not just within one. [Step 11](step_11_tools.ipynb) addresses the first: a tool it can call at runtime to ground its answer in current information. (The third gap — a second, complementary agent — gets its own step later, in [Step 14](step_14_multi_agent_seq.ipynb).)

## Resources for further reading

- Yao, S., et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models*. ICLR 2023. [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)
- Wang, L., et al. (2023). *A Survey on Large Language Model based Autonomous Agents*. [arXiv:2308.11432](https://arxiv.org/abs/2308.11432)
- [CrewAI `Agent` concept docs](https://docs.crewai.com/en/concepts/agents)
- [CrewAI: Crafting Effective Agents](https://docs.crewai.com/en/guides/agents/crafting-effective-agents) — the role/goal/backstory and task description/expected_output guidance the cell above follows, including the "80% of your effort on the task, 20% on the agent" principle

## Stretch goal

Look at the verbose log's "Final Answer" alongside the agent's intermediate reasoning. Find one place where the reasoning and the conclusion seem inconsistent. What does this tell you about trusting chain-of-thought?

---

**→ Your team's Interim Presentation follows after this step (sprint 2).** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for exactly what's expected of the presentation and the design work behind it.